# S3·MIÉ — Un motor de grafo de estados en 40 líneas

**MMIA 6013 · Semana 3 · Miércoles**

Construyes desde cero la abstracción central de LangGraph: estado + nodos +
aristas condicionales + checkpoints. Entender el motor por dentro hace que
LangGraph deje de ser magia.

Todo local, sin dependencias. Al final, el mismo grafo en LangGraph (opcional).

## Copia didáctica resuelta — 30 de septiembre de 2026

Esta copia desarrolla los ejercicios para estudiar. El original se conserva
en `Materiales`. Se mejora la aprobación para distinguir pendiente, aprobación
y rechazo, y se agrega un nodo `rechazar`. El MCP continúa siendo una simulación
en memoria. Las preguntas finales tienen respuestas de ejemplo, no datos de tu
organización. Ejecuta las celdas en orden; no hace falta instalar paquetes.

### Antes del código: qué estás construyendo exactamente

Vas a construir un programa que recibe una pregunta y la hace pasar por varias tareas. Primero implementas un **motor reutilizable**; después le indicas las tareas concretas que debe ejecutar. Por último construyes un cliente y un servidor simulados para publicar y llamar herramientas.

Hay tres momentos distintos al ejecutar el notebook:

1. **Definir:** `class Grafo` y `def nodo_sql(...)` enseñan a Python qué código existe. Todavía no procesan ninguna pregunta.
2. **Configurar:** `g = Grafo()` crea un objeto; `g.agregar_nodo(...)` y `g.agregar_arista(...)` guardan funciones y conexiones en él. Todavía no consultan ni redactan.
3. **Ejecutar:** `g.ejecutar(...)` toma una pregunta, llama las funciones en el orden del grafo y devuelve el resultado.

Es fácil confundirse si mezclas estos momentos. Registrar una función significa «esta tarea está disponible»; ejecutarla significa «haz esta tarea ahora con este estado».

| Celdas del original, contando desde 0 | Qué debes hacer | Qué queda construido |
| --- | --- | --- |
| 1–3 | Importar JSON y completar `Grafo`. | Un motor capaz de recorrer nodos. |
| 4–7 | Definir tareas, registrarlas y conectarlas. | Un flujo para preguntas de ventas, documentos o revisión. |
| 8–10 | Ejecutar hasta la pausa, serializar y restaurar. | Una ejecución que puede continuar con una decisión nueva. |
| 11–12 | Leer la traducción opcional a LangGraph. | Una comparación con una biblioteca. |
| 13–16 | Leer el servidor y completar el cliente. | Un catálogo que puedes descubrir y herramientas que puedes invocar. |
| 17–18 | Agregar un segundo proveedor. | Herramientas de dos fuentes con un despacho común. |
| 19–21 | Justificar la arquitectura y repasar. | Una explicación de cuándo conviene cada mecanismo. |

### Python que necesitas para seguir la práctica

`estado` es un diccionario: una estructura de claves y valores. Su nombre no le da ningún poder especial; podrías llamarlo `expediente` y funcionaría igual.

```python
estado = {"pregunta": "¿Cuánto vendimos en marzo?"}
pregunta = estado["pregunta"]    # Leer un dato existente.
estado["ruta"] = "sql"         # Agregar o reemplazar un dato.
datos = estado.get("datos", {}) # Leer con valor por defecto si falta la clave.
```

Después de la asignación, `estado` contiene la pregunta y la ruta. Leer `estado["datos"]` antes de agregar esa clave causaría `KeyError`; `.get("datos", {})` devuelve `{}` en ese caso. `.get` no agrega esa clave al diccionario.

Una función recibe ese diccionario, trabaja con él y lo devuelve:

```python
def agregar_dato(estado):
    estado["dato"] = 2695
    return estado

expediente = {"pregunta": "ventas de marzo"}
resultado = agregar_dato(expediente)
print(expediente)
print(resultado is expediente)  # True: ambos nombres apuntan al mismo objeto.
```

El resultado impreso contiene `pregunta` y `dato`. **Estos nodos modifican el diccionario que reciben**; `return estado` no crea una copia. Por eso las claves agregadas por un nodo están disponibles para el siguiente. Más adelante, `json.loads` sí producirá un diccionario independiente.



### Qué hace esta importación

`import json` carga funciones para convertir datos Python en texto JSON y restaurarlos. No lee un archivo ni crea un checkpoint por sí sola.


In [1]:
import json

## Parte 1 — El motor

### ✏️ Ejercicio 1.1
Completa la clase `Grafo`:
- `agregar_nodo(nombre, fn)`: fn recibe el estado (dict) y devuelve el estado modificado
- `agregar_arista(desde, hasta)`: transición fija
- `agregar_arista_condicional(desde, router)`: router(estado) → nombre del siguiente nodo
- `ejecutar(estado, inicio, max_pasos=20)`: corre hasta llegar a "FIN",
  registrando en `estado["_traza"]` cada nodo visitado

### Leer la clase sin perderte en `self`

Una clase agrupa datos y operaciones. `g = Grafo()` crea una instancia de esa clase. `__init__` se ejecuta al crearla y prepara sus tres diccionarios vacíos. Dentro de los métodos, `self` se refiere a esa instancia.

```python
g = Grafo()
g.agregar_nodo("sql", nodo_sql)
```

Python entrega `g` como `self`, `"sql"` como `nombre` y la función `nodo_sql` como `fn`. El método realiza `self.nodos[nombre] = fn`: guarda la asociación entre un nombre de texto y una función que podrá llamar después.

`g.agregar_arista("sql", "redactar")` guarda una asociación distinta: del nombre `sql` al nombre `redactar`. No guarda una función, porque una arista fija solo necesita indicar el destino.

La línea central del motor es:

```python
estado = self.nodos[actual](estado)
```

Puedes leerla como tres operaciones más sencillas:

```python
funcion = self.nodos[actual]       # Buscar por nombre la tarea actual.
estado_devuelto = funcion(estado) # Llamarla con el expediente.
estado = estado_devuelto          # Seguir con el expediente que devolvió.
```

Si `actual == "sql"`, la función encontrada es `nodo_sql`. Los primeros corchetes **buscan**; los paréntesis posteriores **llaman**. La palabra `actual` contiene el nombre del nodo que se ejecutará, no el nombre de una variable del dominio ni la pregunta.

### Qué significa cada grupo dentro de `ejecutar`

| Código | Traducción a lenguaje cotidiano | Motivo |
| --- | --- | --- |
| `inicio if inicio is not None else ...` | Si me dieron un punto de inicio, lo uso; si no, leo el guardado. | Servir tanto para comenzar como para reanudar. |
| `estado.setdefault("_traza", [])` | Crea una lista solo si no había una. | Conservar los pasos anteriores al reanudar. |
| `estado["_pausa"] = False` | Empieza esta llamada sin una señal de pausa vieja. | Volver a evaluar la decisión actual. |
| `for _ in range(max_pasos)` | Intenta ejecutar, como máximo, esa cantidad de nodos. | Contener ciclos o recorridos demasiado largos. |
| `if actual == "FIN": break` | Sal del bucle cuando ya no queda una tarea. | No buscar una función llamada `FIN`. |
| `raise ValueError(...)` | Detén la ejecución con un diagnóstico de configuración. | No continuar silenciosamente con un nodo o salida inexistentes. |
| `_traza.append(actual)` | Anota la tarea que acabas de visitar. | Poder reconstruir el recorrido. |
| `if estado.get("_pausa"): ... return estado` | Devuelve inmediatamente el expediente pendiente. | Entregar el control sin ejecutar la próxima tarea. |
| `self.routers[actual](estado)` | Pregunta a la función de ruta qué tarea sigue. | Elegir según el contenido del estado. |
| `self.aristas[actual]` | Lee un destino ya establecido. | Seguir una transición fija. |

El `_` del `for` es una variable que no utilizamos. `range(3)` produce tres iteraciones, con valores 0, 1 y 2. Una iteración aquí ejecuta **un nodo**, no necesariamente una llamada a un modelo.

`break` sale del bucle y permite ejecutar las líneas finales del método. `return estado` sale del método completo y entrega el resultado a quien llamó. En una pausa se necesita `return`, para impedir que el resto del recorrido continúe.

En esta implementación, un router tiene prioridad sobre una arista fija del mismo nodo, porque se consulta primero en el `if`. La configuración de la práctica usa una sola forma de salida por nodo.



In [2]:
class Grafo:
    def __init__(self):
        self.nodos = {}
        self.aristas = {}
        self.routers = {}

    def agregar_nodo(self, nombre, fn):
        self.nodos[nombre] = fn

    def agregar_arista(self, desde, hasta):
        self.aristas[desde] = hasta

    def agregar_arista_condicional(self, desde, router):
        self.routers[desde] = router

    def ejecutar(self, estado, inicio=None, max_pasos=20):
        if type(max_pasos) is not int or max_pasos < 1:
            raise ValueError("max_pasos debe ser un entero positivo")
        actual = inicio if inicio is not None else estado.get("_siguiente", "clasificar")
        estado.setdefault("_traza", [])
        estado["_pausa"] = False
        for _ in range(max_pasos):
            if actual == "FIN":
                break
            if actual not in self.nodos:
                raise ValueError(f"Nodo desconocido: {actual}")
            estado = self.nodos[actual](estado)
            estado["_traza"].append(actual)
            if estado.get("_pausa"):
                estado["_siguiente"] = actual
                estado["_estado_ejecucion"] = "pausado"
                return estado
            if actual in self.routers:
                actual = self.routers[actual](estado)
            elif actual in self.aristas:
                actual = self.aristas[actual]
            else:
                raise ValueError(f"Falta una salida para: {actual}")
            estado["_siguiente"] = actual
        estado["_siguiente"] = actual
        estado["_estado_ejecucion"] = "finalizado" if actual == "FIN" else "limite_pasos"
        return estado


## Parte 2 — El grafo del agente: clasificar → (SQL | RAG) → redactar

Los nodos son simulaciones deterministas (en tu Lab serían llamadas al LLM y
a tus tools reales). Observa que cada nodo SOLO transforma el estado.

### Ver el código de las tareas, no solo sus nombres

En el clasificador, `p` guarda el texto en minúsculas. La expresión con `any` comprueba si se cumple al menos una de estas condiciones:

```python
"ventas" in p
"total" in p
"cuánto" in p
"promedio" in p
```

Para «¿Cuánto vendimos en marzo?», la tercera condición es verdadera. Se escribe `estado["ruta"] = "sql"` y el nodo devuelve el estado. **El clasificador no ejecuta SQL**: deja una etiqueta que luego leerá el router.

Los siguientes tres nodos originales son:

```python
def nodo_sql(estado):
    estado["datos"] = {"total_marzo": 2695, "n_ventas": 4}
    return estado

def nodo_rag(estado):
    estado["datos"] = {
        "chunk": "El trabajo remoto está permitido hasta 3 días por semana."
    }
    return estado

def nodo_redactar(estado):
    estado["respuesta"] = f"[respuesta redactada con {json.dumps(estado.get('datos', {}), ensure_ascii=False)}]"
    return estado
```

En `nodo_sql`, el diccionario de cifras se convierte en el valor de la clave `datos`. En `nodo_rag`, esa misma clave contiene otro diccionario con un fragmento de texto. Se utiliza un lugar común, `datos`, para que `nodo_redactar` pueda trabajar con cualquiera de las dos rutas.

La expresión de redacción se entiende de dentro hacia fuera:

1. `estado.get('datos', {})` obtiene lo que recuperó la ruta, o un objeto vacío.
2. `json.dumps(...)` convierte esos datos en una cadena de texto.
3. La `f` antes de las comillas indica una *f-string*: inserta el valor de la expresión entre llaves en el texto exterior.
4. `estado["respuesta"] = ...` guarda la cadena final en el expediente.
5. `return estado` entrega el expediente que ahora contiene también la respuesta.

Una versión descompuesta de esa misma redacción sería:

```python
datos = estado.get("datos", {})
datos_como_texto = json.dumps(datos, ensure_ascii=False)
respuesta = "[respuesta redactada con " + datos_como_texto + "]"
estado["respuesta"] = respuesta
```

RAG significa recuperar información para usarla al generar una respuesta. El nodo del ejemplo coloca directamente el fragmento que una búsqueda podría haber devuelto. La frase «hasta 3 días» ya está escrita en el código: el programa no la descubrió consultando documentos.



In [3]:
def nodo_clasificar(estado):
    p = estado["pregunta"].lower()
    if any(w in p for w in ("ventas", "total", "cuánto", "promedio")):
        estado["ruta"] = "sql"
    elif any(w in p for w in ("política", "vacaciones", "remoto", "documento")):
        estado["ruta"] = "rag"
    else:
        estado["ruta"] = "sensible"     # lo desconocido pasa por un humano
    return estado

def nodo_sql(estado):
    estado["datos"] = {"total_marzo": 2695, "n_ventas": 4}   # simulación
    return estado

def nodo_rag(estado):
    estado["datos"] = {"chunk": "El trabajo remoto está permitido hasta 3 días por semana."}
    return estado

def nodo_aprobacion(estado):
    # Ausente = pendiente. False = decisión explícita de rechazo.
    if "aprobado" in estado and type(estado["aprobado"]) is not bool:
        raise ValueError("La decisión debe ser True o False")
    estado["_pausa"] = "aprobado" not in estado
    return estado

def nodo_rechazar(estado):
    estado["respuesta"] = "Solicitud rechazada por la persona revisora."
    return estado

def nodo_redactar(estado):
    estado["respuesta"] = f"[respuesta redactada con {json.dumps(estado.get('datos', {}), ensure_ascii=False)}]"
    return estado

### ✏️ Ejercicio 2.1
Arma el grafo con los nodos y las aristas de la parte anterior, y córrelo.

Lo que hay que ver es **de qué habla un grafo que un loop no puede decir**: el estado
viaja explícito entre nodos, y cada arista es una decisión que se puede leer sin ejecutar
nada. Un `while` con `if`s dispersos hace lo mismo y no se puede dibujar.

### Entender `.items()` y reemplazar la `lambda` por una función normal

El diccionario de registro contiene pares como `"sql": nodo_sql`. `.items()` permite recorrer cada par, y `for nombre, fn in ...` separa sus dos partes. Por tanto, la iteración correspondiente a SQL equivale a:

```python
nombre = "sql"
fn = nodo_sql
g.agregar_nodo(nombre, fn)
```

El router escrito como `lambda` hace exactamente lo que haría esta función con nombre:

```python
def decidir_despues_de_clasificar(estado):
    destinos = {
        "sql": "sql",
        "rag": "rag",
        "sensible": "aprobacion",
    }
    ruta_elegida = estado["ruta"]
    siguiente_nodo = destinos[ruta_elegida]
    return siguiente_nodo
```

Para registrar esta versión, podrías usar `g.agregar_arista_condicional("clasificar", decidir_despues_de_clasificar)`. La `lambda` del notebook es una forma corta de expresar esa función.

En la ruta sensible, la etiqueta es `sensible`, pero el nodo se llama `aprobacion`; el diccionario hace esa traducción. En SQL/RAG los nombres coinciden por comodidad, no porque Python exija que una etiqueta y un nodo se llamen igual.

### Recorrer una ejecución mirando las variables del motor

La llamada `g.ejecutar({"pregunta": "¿Cuánto vendimos en marzo?"}, "clasificar")` empieza con `actual="clasificar"` y con la pregunta como única información de dominio:

| Paso | `actual` antes de llamar al nodo | Trabajo realizado | Cómo se elige el siguiente |
| --- | --- | --- | --- |
| 1 | `clasificar` | Agrega `ruta="sql"`. | El router lee esa ruta y devuelve `"sql"`. |
| 2 | `sql` | Agrega las cifras en `datos`. | La arista fija devuelve `"redactar"`. |
| 3 | `redactar` | Agrega el texto en `respuesta`. | La arista fija devuelve `"FIN"`. |

Antes de redactar, la parte del expediente que necesita la función es:

```json
{
  "pregunta": "¿Cuánto vendimos en marzo?",
  "ruta": "sql",
  "datos": {"total_marzo": 2695, "n_ventas": 4}
}
```

El estado completo tiene también las claves de control del motor. La pregunta y las cifras no desaparecen al redactar: la función añade otra clave al mismo diccionario.

El grafo almacena las tareas posibles; `estado` almacena el expediente de una pregunta. Una nueva llamada con un diccionario nuevo puede usar el mismo `g` y conservar un expediente distinto.



In [4]:
g = Grafo()
for nombre, fn in {
    "clasificar": nodo_clasificar,
    "sql": nodo_sql,
    "rag": nodo_rag,
    "aprobacion": nodo_aprobacion,
    "redactar": nodo_redactar,
    "rechazar": nodo_rechazar,
}.items():
    g.agregar_nodo(nombre, fn)

g.agregar_arista_condicional("clasificar", lambda e: {
    "sql": "sql", "rag": "rag", "sensible": "aprobacion"
}[e["ruta"]])
g.agregar_arista("sql", "redactar")
g.agregar_arista("rag", "redactar")
g.agregar_arista_condicional(
    "aprobacion", lambda e: "redactar" if e["aprobado"] else "rechazar"
)
g.agregar_arista("redactar", "FIN")
g.agregar_arista("rechazar", "FIN")

resultado_sql = g.ejecutar({"pregunta": "¿Cuánto vendimos en marzo?"}, "clasificar")
resultado_rag = g.ejecutar({"pregunta": "¿Qué política de trabajo remoto existe?"}, "clasificar")
for resultado in [resultado_sql, resultado_rag]:
    print(json.dumps(resultado, ensure_ascii=False, indent=2))


{
  "pregunta": "¿Cuánto vendimos en marzo?",
  "_traza": [
    "clasificar",
    "sql",
    "redactar"
  ],
  "_pausa": false,
  "ruta": "sql",
  "_siguiente": "FIN",
  "datos": {
    "total_marzo": 2695,
    "n_ventas": 4
  },
  "respuesta": "[respuesta redactada con {\"total_marzo\": 2695, \"n_ventas\": 4}]",
  "_estado_ejecucion": "finalizado"
}
{
  "pregunta": "¿Qué política de trabajo remoto existe?",
  "_traza": [
    "clasificar",
    "rag",
    "redactar"
  ],
  "_pausa": false,
  "ruta": "rag",
  "_siguiente": "FIN",
  "datos": {
    "chunk": "El trabajo remoto está permitido hasta 3 días por semana."
  },
  "respuesta": "[respuesta redactada con {\"chunk\": \"El trabajo remoto está permitido hasta 3 días por semana.\"}]",
  "_estado_ejecucion": "finalizado"
}


## Parte 3 — Human-in-the-loop con checkpoint

Una pregunta "sensible" llega al nodo de aprobación, que **pausa** el grafo.
El estado pausado es serializable (JSON): puede guardarse en disco/DB y
reanudarse horas después — exactamente lo que hace el checkpointer de LangGraph.

### ✏️ Ejercicio 3.1
Ejecuta una pregunta sensible, observa la pausa del checkpoint, y reanuda el grafo con la
decisión humana —una aprobando, otra rechazando—. Anota en qué nodo se detuvo y qué parte
del estado sobrevivió a la pausa.

Lo que hay que notar: el estado se **serializa** en la pausa. Un agente que solo vive en
variables de Python no se puede pausar para esperar a una persona, y esperar a una persona
es el caso normal en cuanto una acción es irreversible — el freno del jueves.

### Leer la serialización y la reanudación línea por línea

`pausado = g.ejecutar(...)` **sí ejecuta** el flujo: visita `clasificar` y `aprobacion`, pero devuelve el control antes de redactar. La variable `pausado` recibe ese diccionario incompleto.

`checkpoint_json = json.dumps(pausado, ensure_ascii=False)` convierte el diccionario en una cadena. Esa cadena contiene una representación de sus claves y valores: `True` de Python se representa como `true` en JSON. No se guarda el bucle ni una función Python en ese texto.

`aprobado = json.loads(checkpoint_json)` lee la cadena y construye otro diccionario. Aquí `aprobado` es el nombre de una variable; `aprobado["aprobado"]` accede a una clave dentro de ese diccionario. Para evitar confundir ambos usos, podrías llamar a la variable `estado_restaurado`:

```python
estado_restaurado = json.loads(checkpoint_json)
estado_restaurado["aprobado"] = True
resultado_final = g.ejecutar(estado_restaurado)
```

Como no se proporciona `inicio`, el motor toma `_siguiente`, que vale `aprobacion`. Limpia `_pausa`, visita ese nodo y descubre que ahora existe la clave `aprobado`. Ya no pide pausa. Solo entonces el router evalúa su valor y envía a `redactar` o `rechazar`.

Esto explica también el orden del código: **se comprueba la pausa antes de llamar al router de aprobación**. Si se llamara al router cuando no existe la clave `aprobado`, `e["aprobado"]` causaría `KeyError`.

Si quieres entender el paso de texto a disco, este bloque opcional escribe y restaura un checkpoint. Depende de haber creado `checkpoint_json` en la celda anterior; crea un archivo en el directorio desde el que ejecutes el notebook:

```python
from pathlib import Path

archivo_checkpoint = Path("checkpoint_ejemplo.json")
archivo_checkpoint.write_text(checkpoint_json, encoding="utf-8")
texto_guardado = archivo_checkpoint.read_text(encoding="utf-8")
estado_desde_disco = json.loads(texto_guardado)
```

Guardar el archivo conserva los datos. Para continuar en otra sesión, aún tendrías que volver a definir la clase, las funciones y las conexiones de `g`. Este bloque es una ampliación explicativa; no se ejecutó al generar las salidas guardadas.



In [5]:
pausado = g.ejecutar({"pregunta": "Autoriza una excepción especial"}, "clasificar")
checkpoint_json = json.dumps(pausado, ensure_ascii=False)
print("CHECKPOINT:", checkpoint_json)

# Dos restauraciones independientes: representan dos decisiones alternativas.
aprobado = json.loads(checkpoint_json)
aprobado["aprobado"] = True
aprobado = g.ejecutar(aprobado)  # Usa _siguiente = aprobacion.

rechazado = json.loads(checkpoint_json)
rechazado["aprobado"] = False
rechazado = g.ejecutar(rechazado)

print("APROBADO:", json.dumps(aprobado, ensure_ascii=False))
print("RECHAZADO:", json.dumps(rechazado, ensure_ascii=False))
print("ORIGINAL SIGUE PAUSADO:", pausado["_estado_ejecucion"])


CHECKPOINT: {"pregunta": "Autoriza una excepción especial", "_traza": ["clasificar", "aprobacion"], "_pausa": true, "ruta": "sensible", "_siguiente": "aprobacion", "_estado_ejecucion": "pausado"}
APROBADO: {"pregunta": "Autoriza una excepción especial", "_traza": ["clasificar", "aprobacion", "aprobacion", "redactar"], "_pausa": false, "ruta": "sensible", "_siguiente": "FIN", "_estado_ejecucion": "finalizado", "aprobado": true, "respuesta": "[respuesta redactada con {}]"}
RECHAZADO: {"pregunta": "Autoriza una excepción especial", "_traza": ["clasificar", "aprobacion", "aprobacion", "rechazar"], "_pausa": false, "ruta": "sensible", "_siguiente": "FIN", "_estado_ejecucion": "finalizado", "aprobado": false, "respuesta": "Solicitud rechazada por la persona revisora."}
ORIGINAL SIGUE PAUSADO: pausado


## Parte 4 (opcional) — El mismo grafo en LangGraph real

Si tienes `pip install langgraph==1.2.12`, la versión que fija el curso, esto es la traducción directa.
Compara: misma semántica, más baterías (checkpointer persistente, streaming,
reintentos, visualización).

### Lectura de la variante opcional del original

Esta celda se conserva como código de lectura: no se instala ni ejecuta LangGraph. El original NO traduce la aprobación: envía `sensible` a `sql`, y `compile()` no recibe un checkpointer. Por eso solo ilustra SQL/RAG; no reproduce la Parte 3. La versión fijada es una indicación del curso, no una recomendación de versión actual.

```python
try:
    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END

    class Estado(TypedDict, total=False):
        pregunta: str
        ruta: str
        datos: dict
        respuesta: str

    sg = StateGraph(Estado)
    sg.add_node("clasificar", nodo_clasificar)
    sg.add_node("sql", nodo_sql)
    sg.add_node("rag", nodo_rag)
    sg.add_node("redactar", nodo_redactar)
    sg.add_edge(START, "clasificar")
    sg.add_conditional_edges("clasificar",
        lambda e: e["ruta"] if e["ruta"] in ("sql", "rag") else "sql",
        {"sql": "sql", "rag": "rag"})
    sg.add_edge("sql", "redactar")
    sg.add_edge("rag", "redactar")
    sg.add_edge("redactar", END)

    app = sg.compile()
    print(app.invoke({"pregunta": "¿Cuánto vendimos en marzo?"})["respuesta"])
except ImportError:
    print("langgraph no instalado — sección de lectura. `pip install langgraph==1.2.12` para probarla.")
```


## Parte 5 — De tools locales a un servidor MCP

El lunes escribiste tres tools y las cableaste a mano dentro de tu loop. Eso
funciona para UNA app. El problema aparece al escalar:

- **M** aplicaciones (tu agente, el IDE, el chat de soporte, un cron)
- **N** herramientas (ventas, RAG, calendario, Jira)
- Sin estándar: **M × N** integraciones, cada una a mano.

MCP (Model Context Protocol) es un protocolo cliente-servidor para ese
problema: cada herramienta se expone UNA vez como servidor, cada aplicación
habla el mismo protocolo como cliente. **M + N** en vez de M × N.

Aquí no instalamos nada: simulamos el servidor en el propio notebook, igual
que el lunes simulamos el LLM. Lo que importa es la **forma** de los mensajes.

### Primero la herramienta, después el servidor

Antes de mirar la clase, observa la función que hace el trabajo:

```python
VENTAS_MES = {"2026-01": 41200, "2026-02": 38750, "2026-03": 52300}

def _consultar_ventas(mes):
    return {"mes": mes, "total": VENTAS_MES.get(mes, 0)}
```

Llamar `_consultar_ventas("2026-03")` devuelve `{"mes": "2026-03", "total": 52300}`. El trabajo útil sigue siendo leer un diccionario. El servidor añadirá una forma común de pedirle ese trabajo.

`_listar_productos()` no recibe argumentos y devuelve `{"productos": PRODUCTOS}`. La lista ya existe en memoria. El guion bajo al inicio de esos nombres es una convención para indicar funciones internas del ejemplo.

### La clase del servidor leída por bloques

En el original, el servidor funciona así:

```python
class ServidorMCP:
    """Servidor MCP simulado. Solo dos métodos: tools/list y tools/call."""

    def __init__(self, nombre):
        self.nombre = nombre
        self._impl = {}
        self._catalogo = []

    def publicar(self, name, description, input_schema, fn):
        self._catalogo.append({"name": name, "description": description,
                               "inputSchema": input_schema})
        self._impl[name] = fn

    def manejar(self, metodo, params=None):
        params = params or {}
        if metodo == "tools/list":
            return {"tools": self._catalogo}
        if metodo == "tools/call":
            nombre = params["name"]
            if nombre not in self._impl:
                return {"isError": True, "content": f"tool desconocida: {nombre}"}
            return {"isError": False,
                    "content": self._impl[nombre](**params.get("arguments", {}))}
        return {"isError": True, "content": f"método no soportado: {metodo}"}
```

| Bloque | Qué guarda o hace |
| --- | --- |
| `__init__(self, nombre)` | Guarda el nombre del servidor y crea sus registros vacíos. |
| `self._impl = {}` | Guarda funciones ejecutables, buscables por nombre. |
| `self._catalogo = []` | Guarda descripciones de herramientas para que el cliente las lea. |
| `publicar(...)` | Añade una descripción a la lista y una función al diccionario. |
| `params = params or {}` | Si no llegaron parámetros, usa un diccionario vacío. |
| Rama `tools/list` | Devuelve las descripciones; no ejecuta herramientas. |
| Rama `tools/call` | Lee el nombre, comprueba que exista, llama la función y envuelve el resultado. |
| Último `return` | Devuelve un error cuando se pide un método que el ejemplo no soporta. |

Los nombres `tools/list` y `tools/call` son cadenas que la función `manejar` compara. En esta simulación no son URLs ni llamadas de red.

### Qué estás diciendo al publicar una herramienta

En `servidor.publicar("consultar_ventas", descripcion, esquema, _consultar_ventas)` entregas cuatro piezas:

1. **Nombre público:** el cliente puede pedir `consultar_ventas` usando ese texto.
2. **Descripción:** explica para qué sirve la herramienta.
3. **Esquema:** describe los argumentos esperados.
4. **Función:** el código real que el servidor llamará.

El nombre público no necesita ser igual al nombre de la función Python. Aquí se publica `consultar_ventas`, mientras la función interna se llama `_consultar_ventas`.

El esquema del mes dice:

```json
{
  "type": "object",
  "properties": {
    "mes": {"type": "string", "description": "Mes YYYY-MM"}
  },
  "required": ["mes"]
}
```

Se lee como «los argumentos son un objeto con una propiedad `mes`, de tipo texto, que debe estar presente». La descripción menciona `YYYY-MM`, pero no impone por sí sola ese patrón. Además, este servidor no utiliza un validador del esquema.

### Desarmar la línea que ejecuta la función por su nombre

```python
nombre = "consultar_ventas"
argumentos = {"mes": "2026-03"}
funcion = servidor._impl[nombre]
resultado = funcion(**argumentos)
```

La última línea equivale a `_consultar_ventas(mes="2026-03")`. Para `listar_productos`, los argumentos son `{}` y `**{}` no agrega ninguno: se llama `_listar_productos()`.

Si la función interna recibe argumentos incorrectos, puede lanzar una excepción antes de que el servidor termine de construir su respuesta. El `isError=False` del ejemplo solo se devuelve si la llamada concluye; no captura automáticamente todas las fallas.



In [6]:
# Un servidor MCP mínimo: publica un catálogo y ejecuta llamadas por nombre.
# Son las MISMAS tres tools del lunes, ahora detrás de un protocolo.
VENTAS_MES = {"2026-01": 41200, "2026-02": 38750, "2026-03": 52300}
PRODUCTOS = ["licencia", "soporte", "capacitación"]

def _consultar_ventas(mes):
    return {"mes": mes, "total": VENTAS_MES.get(mes, 0)}

def _listar_productos():
    return {"productos": PRODUCTOS}

class ServidorMCP:
    """Servidor MCP simulado. Solo dos métodos: tools/list y tools/call."""

    def __init__(self, nombre):
        self.nombre = nombre
        self._impl = {}
        self._catalogo = []

    def publicar(self, name, description, input_schema, fn):
        self._catalogo.append({"name": name, "description": description,
                               "inputSchema": input_schema})
        self._impl[name] = fn

    def manejar(self, metodo, params=None):
        params = params or {}
        if metodo == "tools/list":
            return {"tools": self._catalogo}
        if metodo == "tools/call":
            nombre = params["name"]
            if nombre not in self._impl:
                return {"isError": True, "content": f"tool desconocida: {nombre}"}
            return {"isError": False,
                    "content": self._impl[nombre](**params.get("arguments", {}))}
        return {"isError": True, "content": f"método no soportado: {metodo}"}

servidor = ServidorMCP("ventas-usfq")
servidor.publicar(
    "consultar_ventas",
    "Devuelve el total vendido en un mes dado (formato YYYY-MM).",
    {"type": "object",
     "properties": {"mes": {"type": "string", "description": "Mes YYYY-MM"}},
     "required": ["mes"]},
    _consultar_ventas)
servidor.publicar(
    "listar_productos",
    "Lista los productos distintos que aparecen en las ventas.",
    {"type": "object", "properties": {}},
    _listar_productos)

print(json.dumps(servidor.manejar("tools/list"), ensure_ascii=False, indent=2))

{
  "tools": [
    {
      "name": "consultar_ventas",
      "description": "Devuelve el total vendido en un mes dado (formato YYYY-MM).",
      "inputSchema": {
        "type": "object",
        "properties": {
          "mes": {
            "type": "string",
            "description": "Mes YYYY-MM"
          }
        },
        "required": [
          "mes"
        ]
      }
    },
    {
      "name": "listar_productos",
      "description": "Lista los productos distintos que aparecen en las ventas.",
      "inputSchema": {
        "type": "object",
        "properties": {}
      }
    }
  ]
}


### ✏️ Ejercicio 5.1 — el cliente

Escribe `ClienteMCP`, que es lo que tu agente usaría en lugar del diccionario
`TOOLS` cableado del lunes:

- `descubrir()`: pide `tools/list` y guarda el catálogo. **El cliente no sabe
  de antemano qué herramientas hay** — esa es la diferencia con el lunes.
- `esquemas_para_llm()`: devuelve el catálogo en el formato que espera la API
  del modelo (`name`, `description`, `input_schema`).
- `invocar(nombre, argumentos)`: hace `tools/call` y devuelve el contenido.

### Qué ocurre en el cliente durante una llamada

`cliente = ClienteMCP(servidor)` guarda una referencia al objeto `servidor` en `self.servidor`. Así el cliente sabe **con quién hablar**. Todavía no sabe qué herramientas publica, porque su catálogo empieza como `[]`.

En `descubrir()`, `self.servidor.manejar("tools/list")` pide las descripciones. `respuesta["tools"]` extrae la lista del objeto recibido. El paso `json.loads(json.dumps(...))` crea una copia de los datos del catálogo; cambiar la copia del cliente no cambia directamente la lista del servidor. Esa copia es una elección de la práctica, no un requisito de descubrimiento.

La comprensión de lista en `esquemas_para_llm()` recorre las herramientas. Puedes leerla como este bucle, que produce el mismo tipo de lista:

```python
esquemas = []
for herramienta in cliente.catalogo:
    esquema = {
        "name": herramienta["name"],
        "description": herramienta["description"],
        "input_schema": herramienta["inputSchema"],
    }
    esquemas.append(esquema)
```

Cada `herramienta` es una descripción, no la función Python que implementa el trabajo. Este paso prepara información para describir herramientas al modelo; no genera una respuesta ni llama al modelo.

Ahora sigue `cliente.invocar("consultar_ventas", {"mes": "2026-03"})`:

1. Comprueba que los argumentos sean un diccionario. Esto no comprueba todavía que tenga las claves o los tipos exigidos por cada herramienta.
2. Construye `{"name": "consultar_ventas", "arguments": {"mes": "2026-03"}}`.
3. Llama `manejar("tools/call", ...)` en el servidor.
4. Recibe la envoltura con `isError` y `content`.
5. Si es un error, lanza `RuntimeError`; si no, devuelve solo `content`.

Las últimas llamadas a `print` muestran datos devueltos por funciones reales de Python con datos ficticios. No muestran una decisión autónoma de un LLM.



In [7]:
class ClienteMCP:
    def __init__(self, servidor):
        self.servidor = servidor
        self.catalogo = []

    def descubrir(self):
        respuesta = self.servidor.manejar("tools/list")
        self.catalogo = json.loads(json.dumps(respuesta["tools"]))
        return self.catalogo

    def esquemas_para_llm(self):
        # Formato INTERNO del ejercicio; un proveedor puede pedir otro envoltorio.
        return [{"name": t["name"], "description": t["description"],
                 "input_schema": t["inputSchema"]} for t in self.catalogo]

    def invocar(self, nombre, argumentos):
        if not isinstance(argumentos, dict):
            raise TypeError("Los argumentos deben ser un diccionario")
        resultado = self.servidor.manejar("tools/call", {
            "name": nombre, "arguments": argumentos
        })
        if resultado.get("isError"):
            raise RuntimeError(resultado["content"])
        return resultado["content"]

cliente = ClienteMCP(servidor)
cliente.descubrir()
print("CATÁLOGO PARA EL MODELO:", json.dumps(cliente.esquemas_para_llm(), ensure_ascii=False))
print("VENTAS:", cliente.invocar("consultar_ventas", {"mes": "2026-03"}))
print("PRODUCTOS:", cliente.invocar("listar_productos", {}))


CATÁLOGO PARA EL MODELO: [{"name": "consultar_ventas", "description": "Devuelve el total vendido en un mes dado (formato YYYY-MM).", "input_schema": {"type": "object", "properties": {"mes": {"type": "string", "description": "Mes YYYY-MM"}}, "required": ["mes"]}}, {"name": "listar_productos", "description": "Lista los productos distintos que aparecen en las ventas.", "input_schema": {"type": "object", "properties": {}}}]
VENTAS: {'mes': '2026-03', 'total': 52300}
PRODUCTOS: {'productos': ['licencia', 'soporte', 'capacitación']}


### ✏️ Ejercicio 5.2 — la cuenta que justifica el estándar

Un segundo servidor entra en juego y el agente no cambia ni una línea: solo
descubre más herramientas. Eso es el desacople.

### Qué añade exactamente el gestor de dos clientes

`GestorMCP` es una ampliación didáctica de la solución. Resuelve una pregunta concreta: cuando hay dos clientes, **¿a cuál le mando una petición?**

```python
gestor = GestorMCP({
    "ventas": cliente,
    "rrhh": ClienteMCP(servidor_rrhh),
})
```

El diccionario entrega un alias y un cliente para cada proveedor. Al descubrir, el gestor visita cada cliente, pide su catálogo y construye un nombre con prefijo:

```python
nombre = f"{alias}__{tool['name']}"
self.rutas[nombre] = (cliente, tool["name"])
```

Si el alias es `rrhh` y el nombre local es `consultar_politica`, el nombre agregado es `rrhh__consultar_politica`. El valor guardado en `rutas` contiene **el cliente correspondiente y el nombre original**. Es una pareja o tupla de dos elementos.

La invocación deshace esa traducción:

```python
cliente, nombre_original = self.rutas[nombre]
return cliente.invocar(nombre_original, argumentos)
```

El servidor de RR. HH. recibe `consultar_politica`, sin el prefijo. El prefijo pertenece al catálogo agrupado de nuestra aplicación.

En `self.catalogo.append({**tool, "name": nombre})`, el uso de `**tool` tiene otro significado contextual: **copia las claves del diccionario** en uno nuevo. Después `"name": nombre` sustituye el nombre local por el nombre con prefijo. En una llamada como `funcion(**argumentos)`, en cambio, `**` reparte argumentos nombrados.

El servidor añadido usa `lambda: {"remoto_dias_por_semana": 3}`: una función sin argumentos que devuelve ese diccionario. Es la forma breve de definir una herramienta fija de RR. HH. La lógica de selección de políticas no forma parte de este ejemplo.



In [8]:
# Un segundo proveedor, con datos ficticios, usa la misma interfaz.
servidor_rrhh = ServidorMCP("rrhh-simulado")
servidor_rrhh.publicar(
    "consultar_politica", "Devuelve la política simulada de trabajo remoto.",
    {"type": "object", "properties": {}},
    lambda: {"remoto_dias_por_semana": 3}
)

class GestorMCP:
    def __init__(self, clientes):
        # Las claves son identificadores únicos elegidos por nuestro host.
        self.clientes = clientes
        self.catalogo = []
        self.rutas = {}

    def descubrir(self):
        self.catalogo = []
        self.rutas = {}
        for alias, cliente in self.clientes.items():
            cliente.descubrir()
            for tool in cliente.esquemas_para_llm():
                nombre = f"{alias}__{tool['name']}"
                if nombre in self.rutas:
                    raise ValueError(f"Nombre duplicado: {nombre}")
                self.rutas[nombre] = (cliente, tool["name"])
                self.catalogo.append({**tool, "name": nombre})
        return self.catalogo

    def esquemas_para_llm(self):
        return self.catalogo

    def invocar(self, nombre, argumentos):
        cliente, nombre_original = self.rutas[nombre]
        return cliente.invocar(nombre_original, argumentos)

gestor = GestorMCP({"ventas": cliente, "rrhh": ClienteMCP(servidor_rrhh)})
gestor.descubrir()

# El controlador conserva la misma lógica para uno o varios proveedores.
# La petición aquí está escrita por nosotros, no producida por un LLM.
def ejecutar_peticion(toolset, peticion):
    return toolset.invocar(peticion["name"], peticion["arguments"])

print("TOOLS:", [t["name"] for t in gestor.esquemas_para_llm()])
print("VENTAS:", ejecutar_peticion(gestor, {
    "name": "ventas__consultar_ventas", "arguments": {"mes": "2026-03"}
}))
print("RRHH:", ejecutar_peticion(gestor, {
    "name": "rrhh__consultar_politica", "arguments": {}
}))


TOOLS: ['ventas__consultar_ventas', 'ventas__listar_productos', 'rrhh__consultar_politica']
VENTAS: {'mes': '2026-03', 'total': 52300}
RRHH: {'remoto_dias_por_semana': 3}


In [9]:
# Ejemplo ficticio; cambia M y N por los de tu organización.
M, N = 4, 6
print("Integraciones directas:", M * N)  # 24
print("Componentes con protocolo:", M + N)  # 10
# M×N cuenta enlaces particulares; M+N cuenta adaptadores/componentes.
# MCP no decide por sí solo permisos ni calidad de la respuesta, y el host
# aún debe ejecutar el despacho y aplicar su política. La cuenta no mide costo.


Integraciones directas: 24
Componentes con protocolo: 10


In [10]:
# Flujo de ejemplo (no entrega académica):
# clasificar -> sql/rag/aprobacion -> redactar/rechazar -> FIN
# Elijo grafo porque tengo rutas distintas y un punto explícito de pausa.
# Puedo guardar el estado con el nodo de reanudación y revisar la traza.
# Para una secuencia breve y fija sin pausas, un loop simple sería suficiente.


## Cierre

- Grafo de estados = nodos (funciones) + aristas condicionales + estado compartido.
- Checkpoint = estado serializable + punto de reanudación → human-in-the-loop limpio.
- LangGraph te da esto con baterías; ahora sabes qué hay debajo.
- MCP: un servidor publica `tools/list`, un cliente invoca `tools/call`. El agente
  descubre herramientas en vez de tenerlas cableadas → M + N en vez de M × N.
- MCP no reemplaza a function calling: function calling es cómo el modelo PIDE,
  MCP es cómo la herramienta se PUBLICA. Y ejecutar sigue siendo cosa de tu código.

**Mañana:** todo lo que puede salir mal — riesgos, costos descontrolados,
inyección vía tools — y la ética de los agentes autónomos.

## Verificación local
Estas comprobaciones usan únicamente los ejemplos en memoria.


### Cómo leer las comprobaciones

`assert condicion` detiene la ejecución si la condición es falsa. Aquí comprueba recorridos, datos y estados esperados; no escribe la solución. Un `try/except` permite comprobar que una petición incorrecta produce el error esperado. El diccionario `resultados_verificados` reúne salidas para inspeccionarlas.


In [11]:
# Comprobaciones de conducta: rutas, pausa/restauración, límites y despacho.
assert resultado_sql["_traza"] == ["clasificar", "sql", "redactar"]
assert resultado_sql["datos"]["total_marzo"] == 2695
assert resultado_rag["_traza"] == ["clasificar", "rag", "redactar"]
assert pausado["_siguiente"] == "aprobacion" and pausado["_pausa"]
assert "respuesta" not in pausado
assert aprobado["_traza"] == ["clasificar", "aprobacion", "aprobacion", "redactar"]
assert rechazado["_traza"] == ["clasificar", "aprobacion", "aprobacion", "rechazar"]
assert aprobado["_estado_ejecucion"] == rechazado["_estado_ejecucion"] == "finalizado"
assert aprobado["pregunta"] == rechazado["pregunta"] == pausado["pregunta"]
assert g.ejecutar(json.loads(checkpoint_json))["_estado_ejecucion"] == "pausado"
limite = g.ejecutar({"pregunta": "total de ventas"}, "clasificar", max_pasos=1)
assert limite["_estado_ejecucion"] == "limite_pasos" and "respuesta" not in limite
assert g.ejecutar(limite)["_estado_ejecucion"] == "finalizado"
assert g.ejecutar({"pregunta": "total de ventas"}, "clasificar", max_pasos=3)["_estado_ejecucion"] == "finalizado"
bucle = Grafo()
bucle.agregar_nodo("bucle", lambda e: e)
bucle.agregar_arista("bucle", "bucle")
assert bucle.ejecutar({}, "bucle", max_pasos=2)["_estado_ejecucion"] == "limite_pasos"
assert cliente.invocar("consultar_ventas", {"mes": "2026-03"})["total"] == 52300
assert cliente.invocar("listar_productos", {})["productos"] == ["licencia", "soporte", "capacitación"]
assert gestor.invocar("rrhh__consultar_politica", {})["remoto_dias_por_semana"] == 3
assert len(gestor.catalogo) == 3
for nombre, argumentos, error in [
    ("no_existe", {}, RuntimeError),
    ("consultar_ventas", {}, TypeError),
    ("consultar_ventas", [], TypeError),
]:
    try:
        cliente.invocar(nombre, argumentos)
    except error:
        pass
    else:
        raise AssertionError(f"No se detectó el error de {nombre}")
# Dos proveedores pueden publicar el mismo nombre local sin confundirse.
otro = ServidorMCP("tambien-ventas")
otro.publicar("consultar_ventas", "Otra fuente ficticia", {"type": "object"}, lambda mes: {"total": 99})
separado = GestorMCP({"uno": cliente, "dos": ClienteMCP(otro)})
separado.descubrir()
assert separado.invocar("uno__consultar_ventas", {"mes": "2026-03"})["total"] == 52300
assert separado.invocar("dos__consultar_ventas", {"mes": "2026-03"})["total"] == 99
resultados_verificados = {
    "sql": resultado_sql, "rag": resultado_rag, "checkpoint": pausado,
    "aprobacion": aprobado, "rechazo": rechazado,
    "tools_descubiertas": [t["name"] for t in gestor.catalogo],
    "mcp_ventas_marzo": cliente.invocar("consultar_ventas", {"mes": "2026-03"}),
    "verificacion": "Todas las comprobaciones locales pasaron",
    "langgraph_opcional": "No ejecutado; solo lectura",
}
print(resultados_verificados["verificacion"])


Todas las comprobaciones locales pasaron
